In [1]:
# ======================================================================
# CLEAN DUBLIN BER: one ML-ready table, one row per Dublin home
# ======================================================================
# Input:  the 19 Dublin processed BER tables
#         (just_dublin_dataset\just_dublin_processed_dataset\ber)
# Output: just_dublin_dataset\Just_dublin_clean_dataset\ber\ber_dublin_clean.csv
#
# Same cleaning as the national BER notebook, but for Dublin only and
# keeping the location columns:
#   nation, province, county, local_authority (empty for BER), dublin_district
#
# What this table is for (Student 1):
#   - K-Prototypes clustering + FAMD   -> building features
#   - XGBoost regression               -> target: BerRating (kWh/m2/yr)
#   - Random Forest classification     -> target: is_B3_or_better
#
# Rules: fix the data, do NOT fill missing values / scale / encode here,
# and leave out leakage columns (outputs of the BER calculation).
#
# Needs the Dublin processed notebooks (00 and 01) to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

ber_folder   = base + r'\just_dublin_dataset\just_dublin_processed_dataset\ber'
dublin_clean = base + r'\just_dublin_dataset\Just_dublin_clean_dataset'

In [2]:
# Loading only the tables that hold useful inputs or the targets.
# Left out on purpose:
#   group_heating, group_boilers, other_energy, solar_pv, wind_turbines, floors
#   -> mostly empty or already summed up in other columns
#   most of results -> CO2Rating, EPC, CPC, RER, primary energy columns,
#   HeatLossIndicator are calculated from the same energy calculation as
#   BerRating, so using them as inputs would be leakage

def load(name):
    return pd.read_csv(ber_folder + '\\dublin_' + name + '.csv', low_memory=False)

assessment    = load('assessment')
results       = load('results')[['ber_id', 'BerRating', 'EnergyRating']]
geometry      = load('geometry')
fabric        = load('fabric')
ventilation   = load('ventilation')
space_heating = load('space_heating')
water_heating = load('water_heating')
solar_water   = load('solar_water')[['ber_id', 'SolarHotWaterHeating']]
renewables    = load('nzeb_renewables')[['ber_id', 'SolarPVEnergyOutput']]
walls         = load('walls')

ber_county    = load('county')
dwelling_type = load('dwelling_type')
fuel          = load('fuel')

print('assessment:', assessment.shape)
print('walls:     ', walls.shape)

assessment: (181195, 19)
walls:      (315354, 9)


In [3]:
# Picking the columns that describe the building itself.
# These are the things an assessor records about the house: size, insulation,
# ventilation, heating system. None of them are results of the rating.

# The location columns made in the Dublin processed notebook are kept too.

assessment = assessment[['ber_id', 'nation', 'province', 'county', 'county_id', 'local_authority',
                         'dublin_district', 'district_id', 'ber_location',
                         'dwelling_type_id', 'Year_of_Construction',
                         'TypeofRating', 'PurposeOfRating', 'DateOfAssessment']]

geometry = geometry[['ber_id', 'GroundFloorArea(sq m)', 'NoStoreys', 'WallArea', 'RoofArea',
                     'FloorArea', 'WindowArea', 'DoorArea', 'LivingAreaPercent',
                     'RoomInRoofArea', 'PredominantRoofType']]

fabric = fabric[['ber_id', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor',
                 'ThermalBridgingFactor', 'ThermalMassCategory', 'StructureType',
                 'SuspendedWoodenFloor', 'PercentageDraughtStripped', 'NoOfSidesSheltered',
                 'PermeabilityTest']]

ventilation = ventilation[['ber_id', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents',
                           'NoOfFluelessGasFires', 'DraftLobby', 'VentilationMethod',
                           'HeatExchangerEff']]

space_heating = space_heating[['ber_id', 'main_fuel_id', 'HSMainSystemEfficiency',
                               'HSSupplHeatFraction', 'HeatSystemControlCat',
                               'HeatSystemResponseCat', 'UndergroundHeating',
                               'WarmAirHeatingSystem', 'CombiBoiler']]
space_heating = space_heating.rename(columns={'main_fuel_id': 'space_fuel_id'})

water_heating = water_heating[['ber_id', 'main_fuel_id', 'WHMainSystemEff', 'WaterStorageVolume',
                               'InsulationType', 'CylinderStat']]
water_heating = water_heating.rename(columns={'main_fuel_id': 'water_fuel_id'})

In [4]:
# A house can have up to 3 wall types. For one row per BER, keeping the
# main wall type = the one with the biggest area.

walls = walls.sort_values(['ber_id', 'wall_area'], ascending=[True, False])
main_wall = walls.drop_duplicates(subset='ber_id', keep='first')
main_wall = main_wall[['ber_id', 'wall_type']].rename(columns={'wall_type': 'MainWallType'})

print('BERs with a main wall type:', len(main_wall))
main_wall.head()

BERs with a main wall type: 181195


,ber_id,MainWallType
0,14,300mm Filled Cavity
1,19,Unknown
2,24,Unknown
3,25,Unknown
4,27,Unknown


In [5]:
# Joining everything back into one table using ber_id (like a SQL join).
# how='left' keeps every BER from the assessment table.

ber = assessment.merge(results, on='ber_id', how='left')
ber = ber.merge(geometry, on='ber_id', how='left')
ber = ber.merge(fabric, on='ber_id', how='left')
ber = ber.merge(ventilation, on='ber_id', how='left')
ber = ber.merge(space_heating, on='ber_id', how='left')
ber = ber.merge(water_heating, on='ber_id', how='left')
ber = ber.merge(solar_water, on='ber_id', how='left')
ber = ber.merge(renewables, on='ber_id', how='left')
ber = ber.merge(main_wall, on='ber_id', how='left')

print(ber.shape)   # should still be 569,058 rows

(181195, 61)


In [6]:
# Putting the names back in place of the ids (dwelling type and fuels),
# so the table is readable and the models can encode them later.

type_map = dict(zip(dwelling_type['dwelling_type_id'], dwelling_type['DwellingTypeDescr']))
fuel_map = dict(zip(fuel['fuel_id'], fuel['FuelName']))

ber['DwellingType']         = ber['dwelling_type_id'].map(type_map)
ber['MainSpaceHeatingFuel'] = ber['space_fuel_id'].map(fuel_map)
ber['MainWaterHeatingFuel'] = ber['water_fuel_id'].map(fuel_map)

ber = ber.drop(columns=['dwelling_type_id', 'space_fuel_id', 'water_fuel_id'])

print(ber['DwellingType'].value_counts(dropna=False))
print(ber['MainSpaceHeatingFuel'].value_counts(dropna=False))

DwellingType
Semi-detached house       42443
Mid-floor apartment       40806
Mid-terrace house         30588
End of terrace house      16431
Ground-floor apartment    16145
Top-floor apartment       16109
Detached house            14854
Maisonette                 3689
Basement Dwelling           128
Apartment                     2
Name: count, dtype: int64
MainSpaceHeatingFuel
Mains Gas                         88411
Electricity                       71649
NaN                               11334
Heating Oil                        9191
Manufactured Smokeless Fuel         447
Bulk LPG (propane or butane)         77
Solid Multi-Fuel                     29
Wood Pellets (bulk supply for        23
Bottled LPG                          16
House Coal                            5
Wood Logs                             4
Wood Chips                            3
Wood Pellets (in bags for seco        3
Anthracite                            2
Biodiesel from renewable sourc        1
Name: count, dtype: 

In [7]:
# Location check: every Dublin home should have a district
# (a real one, or 'Dublin (no district)' for homes recorded as 'Co. Dublin').

print('Rows without a district:', ber['dublin_district'].isnull().sum())   # should be 0
print(ber['dublin_district'].value_counts())

Rows without a district: 0
dublin_district
Dublin (no district)    48327
Dublin 15               13465
Dublin 18               12075
Dublin 24               11841
Dublin 12                7457
Dublin 8                 7260
Dublin 11                7119
Dublin 22                6906
Dublin 9                 6903
Dublin 7                 6265
Dublin 13                5829
Dublin 4                 5789
Dublin 16                5670
Dublin 14                5362
Dublin 3                 5120
Dublin 5                 5034
Dublin 1                 4556
Dublin 6                 4106
Dublin 6W                2808
Dublin 10                2631
Dublin 17                2363
Dublin 2                 2246
Dublin 20                2063
Name: count, dtype: int64


In [8]:
# Text columns: SEAI pads a lot of values with spaces at the end
# ('Existing       ', 'NO  ', 'Masonry   '). Removing them.
# Also some 'missing' values are hidden as text: '' or 'Please select'.
# Turning those into real missing values (NaN).

text_cols = []
for col in ber.columns:
    if ber[col].dtype == 'object' or str(ber[col].dtype) == 'str':
        text_cols.append(col)

for col in text_cols:
    ber[col] = ber[col].str.strip()
    ber[col] = ber[col].replace(['', 'Please select'], np.nan)

print('Text columns cleaned:', text_cols)

Text columns cleaned: ['nation', 'province', 'county', 'dublin_district', 'ber_location', 'TypeofRating', 'PurposeOfRating', 'DateOfAssessment', 'EnergyRating', 'PredominantRoofType', 'ThermalMassCategory', 'StructureType', 'SuspendedWoodenFloor', 'PermeabilityTest', 'DraftLobby', 'VentilationMethod', 'UndergroundHeating', 'WarmAirHeatingSystem', 'CombiBoiler', 'InsulationType', 'CylinderStat', 'SolarHotWaterHeating', 'MainWallType', 'DwellingType', 'MainSpaceHeatingFuel', 'MainWaterHeatingFuel']


In [9]:
# YES / NO columns -> 1 / 0, so models can use them directly.
# Only changing columns where every value is YES, NO, Yes or No.

yes_no_cols = []
for col in text_cols:
    values = set(ber[col].dropna().unique())
    if len(values) > 0 and values.issubset({'YES', 'NO', 'Yes', 'No'}):
        ber[col] = ber[col].map({'YES': 1, 'Yes': 1, 'NO': 0, 'No': 0})
        yes_no_cols.append(col)

print('Changed to 1/0:', yes_no_cols)

Changed to 1/0: ['PermeabilityTest', 'DraftLobby', 'UndergroundHeating', 'WarmAirHeatingSystem', 'CylinderStat', 'SolarHotWaterHeating']


In [10]:
# Dates come back as text from CSV, so converting again.
# Adding year and quarter (e.g. 2021Q1) in the same format as PPR, rent and
# planning, for the county + quarter join later.

ber['DateOfAssessment'] = pd.to_datetime(ber['DateOfAssessment'])
ber['year'] = ber['DateOfAssessment'].dt.year
ber['quarter'] = ber['DateOfAssessment'].dt.to_period('Q').astype(str)

print(ber['year'].value_counts().sort_index())

year
2021    24029
2022    32131
2023    36985
2024    42893
2025    45157
Name: count, dtype: int64


In [11]:
# Year of construction goes up to 2073, which is impossible.
# A few 2026 values are fine (provisional ratings for homes not built yet),
# so only years after 2026 become missing.

print('Years after 2026:', (ber['Year_of_Construction'] > 2026).sum())
print(ber[ber['Year_of_Construction'] > 2026][['ber_id', 'Year_of_Construction', 'TypeofRating']])

ber.loc[ber['Year_of_Construction'] > 2026, 'Year_of_Construction'] = np.nan

Years after 2026: 0
Empty DataFrame
Columns: [ber_id, Year_of_Construction, TypeofRating]
Index: []


In [12]:
# Looking at the energy rating labels after removing spaces.
# Expected the usual 15 bands (A1 ... G), but there are also A, A0, B, C, D, E.

print(ber['EnergyRating'].value_counts().sort_index())

EnergyRating
A       109
A0       53
A1     4880
A2    48806
A3    10867
B        59
B1     6050
B2    11836
B3    18568
C        33
C1    16173
C2    14827
C3    12356
D        17
D1    10721
D2     9063
E         1
E1     5409
E2     3981
F      3812
G      3574
Name: count, dtype: int64


In [13]:
# The odd labels only show up from 2021 and grow a lot in 2024-2025,
# and their BerRating values sit in clear ranges (A0 lowest, then A, B, C...).
# So they look like a different (newer) rating scale, not typing mistakes.
# Keeping the rows, but marking which scale each label belongs to.

old_scale = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3',
             'D1', 'D2', 'E1', 'E2', 'F', 'G']

ber['rating_scale'] = 'other'
ber.loc[ber['EnergyRating'].isin(old_scale), 'rating_scale'] = 'old_15_band'

print(ber['rating_scale'].value_counts())
print()
print(ber.groupby(['EnergyRating'])['BerRating'].agg(['count', 'min', 'max']).sort_values('min'))

rating_scale
old_15_band    180923
other             272
Name: count, dtype: int64

              count     min      max
EnergyRating                        
A1             4880 -132.61    25.00
A0               53  -14.53    42.00
A               109    1.28    74.99
A2            48806   25.00    50.00
A3            10867   50.00    75.00
B1             6050   75.00   100.00
B                59   79.97   149.50
B2            11836  100.01   125.00
B3            18568  125.00   150.00
C1            16173  150.00   175.00
C                33  154.67   215.60
C2            14827  175.00   200.00
C3            12356  200.00   225.00
D1            10721  225.01   260.00
D                17  227.27   270.02
D2             9063  260.00   299.99
E                 1  282.06   282.06
E1             5409  300.02   339.98
E2             3981  340.01   379.98
F              3812  374.60   449.90
G              3574  423.88  1918.41


In [14]:
# Classification target: is the home B3 or better?
# Homes on the 15-band scale: A1, A2, A3, B1, B2, B3 -> 1, everything else -> 0
# (using the letters here, because a few homes sit exactly on 150 and the
#  letter is the official answer for them)
# Homes on the other scale (A0, A, B, C, D, E) have no B3 letter, so for those
# the BerRating cut-off is used instead: B3 or better = up to 150 kWh/m2/yr.

b3_or_better = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3']

ber['is_B3_or_better'] = 0
ber.loc[ber['EnergyRating'].isin(b3_or_better), 'is_B3_or_better'] = 1
ber.loc[(ber['rating_scale'] == 'other') & (ber['BerRating'] <= 150), 'is_B3_or_better'] = 1

# Checking the labels
print(pd.crosstab(ber['EnergyRating'], ber['is_B3_or_better']))
print()
print(ber['is_B3_or_better'].value_counts(normalize=True).round(3))

is_B3_or_better      0      1
EnergyRating                 
A                    0    109
A0                   0     53
A1                   0   4880
A2                   0  48806
A3                   0  10867
B                    0     59
B1                   0   6050
B2                   0  11836
B3                   0  18568
C                   33      0
C1               16173      0
C2               14827      0
C3               12356      0
D                   17      0
D1               10721      0
D2                9063      0
E                    1      0
E1                5409      0
E2                3981      0
F                 3812      0
G                 3574      0

is_B3_or_better
1    0.559
0    0.441
Name: proportion, dtype: float64


In [15]:
# BerRating range check. Negative values are homes that make more energy than
# they use (e.g. solar panels), so they are real and kept.
# Very high values are kept too; outliers are dealt with at the modelling stage.

print(ber['BerRating'].describe().round(2))
print('Negative BerRating:', (ber['BerRating'] < 0).sum())

count    181195.00
mean        149.58
std         117.97
min        -132.61
25%          44.60
50%         137.44
75%         207.59
max        1918.41
Name: BerRating, dtype: float64
Negative BerRating: 536


In [16]:
# Tidying column names and order: keys first, then building features,
# then the targets at the end.

ber = ber.rename(columns={'GroundFloorArea(sq m)': 'TotalFloorArea'})

key_cols = ['ber_id', 'nation', 'province', 'county', 'county_id', 'local_authority',
            'dublin_district', 'district_id', 'ber_location', 'DateOfAssessment', 'year', 'quarter']
target_cols = ['BerRating', 'EnergyRating', 'rating_scale', 'is_B3_or_better']

feature_cols = []
for col in ber.columns:
    if col not in key_cols and col not in target_cols:
        feature_cols.append(col)

ber = ber[key_cols + feature_cols + target_cols]

print(len(feature_cols), 'feature columns:')
print(feature_cols)

49 feature columns:
['Year_of_Construction', 'TypeofRating', 'PurposeOfRating', 'TotalFloorArea', 'NoStoreys', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'LivingAreaPercent', 'RoomInRoofArea', 'PredominantRoofType', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor', 'ThermalBridgingFactor', 'ThermalMassCategory', 'StructureType', 'SuspendedWoodenFloor', 'PercentageDraughtStripped', 'NoOfSidesSheltered', 'PermeabilityTest', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents', 'NoOfFluelessGasFires', 'DraftLobby', 'VentilationMethod', 'HeatExchangerEff', 'HSMainSystemEfficiency', 'HSSupplHeatFraction', 'HeatSystemControlCat', 'HeatSystemResponseCat', 'UndergroundHeating', 'WarmAirHeatingSystem', 'CombiBoiler', 'WHMainSystemEff', 'WaterStorageVolume', 'InsulationType', 'CylinderStat', 'SolarHotWaterHeating', 'SolarPVEnergyOutput', 'MainWallType', 'DwellingType', 'MainSpaceHeatingFuel', 'MainWaterHeatingFuel']


In [17]:
# Missing values left in the clean table (% empty per column).
# Not filling them here; the model Pipeline will handle them on training data.

missing = (ber.isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

local_authority              100.0
PredominantRoofType           27.1
StructureType                 13.9
SuspendedWoodenFloor           9.4
PercentageDraughtStripped      9.4
NoOfSidesSheltered             9.4
PermeabilityTest               9.4
NoOfChimneys                   9.4
NoOfOpenFlues                  9.4
NoOfFansAndVents               9.4
NoOfFluelessGasFires           9.4
DraftLobby                     9.4
VentilationMethod              9.4
HeatExchangerEff               9.4
HSMainSystemEfficiency         6.3
HSSupplHeatFraction            6.3
WHMainSystemEff                6.3
MainSpaceHeatingFuel           6.3
MainWaterHeatingFuel           6.3
dtype: float64

In [18]:
ber.info()
ber.head()

<class 'pandas.DataFrame'>
RangeIndex: 181195 entries, 0 to 181194
Data columns (total 65 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   ber_id                     181195 non-null  int64         
 1   nation                     181195 non-null  str           
 2   province                   181195 non-null  str           
 3   county                     181195 non-null  str           
 4   county_id                  181195 non-null  int64         
 5   local_authority            0 non-null       float64       
 6   dublin_district            181195 non-null  str           
 7   district_id                181195 non-null  int64         
 8   ber_location               181195 non-null  str           
 9   DateOfAssessment           181195 non-null  datetime64[us]
 10  year                       181195 non-null  int32         
 11  quarter                    181195 non-null  str           
 12 

,ber_id,nation,province,county,county_id,local_authority,dublin_district,district_id,ber_location,DateOfAssessment,year,quarter,Year_of_Construction,TypeofRating,PurposeOfRating,TotalFloorArea,NoStoreys,WallArea,RoofArea,FloorArea,WindowArea,DoorArea,LivingAreaPercent,RoomInRoofArea,PredominantRoofType,UValueWall,UValueRoof,UValueFloor,UValueWindow,UvalueDoor,ThermalBridgingFactor,ThermalMassCategory,StructureType,SuspendedWoodenFloor,PercentageDraughtStripped,NoOfSidesSheltered,PermeabilityTest,NoOfChimneys,NoOfOpenFlues,NoOfFansAndVents,NoOfFluelessGasFires,DraftLobby,VentilationMethod,HeatExchangerEff,HSMainSystemEfficiency,HSSupplHeatFraction,HeatSystemControlCat,HeatSystemResponseCat,UndergroundHeating,WarmAirHeatingSystem,CombiBoiler,WHMainSystemEff,WaterStorageVolume,InsulationType,CylinderStat,SolarHotWaterHeating,SolarPVEnergyOutput,MainWallType,DwellingType,MainSpaceHeatingFuel,MainWaterHeatingFuel,BerRating,EnergyRating,rating_scale,is_B3_or_better
0,14,Ireland,Leinster,Dublin,6,NaN,Dublin 12,13,Dublin 12,2021-01-03,2021,2021Q1,1930.0,Existing,Grant Support,67.050,1,54.27,67.05,67.05,9.21,2.18,35.44,0.0,Pitch Roof-Insul.on Ceiling,0.29,0.19,0.73,1.10,1.71,0.15,High,Masonry,No,88.0,2.0,0.0,0.0,1.0,2.0,0.0,0.0,Natural vent.,0.0,91.1,0.1,1,1,0,0,"Instantaneous, without keep-ho",91.1,0.0,None,0,0,0.0,300mm Filled Cavity,Semi-detached house,Mains Gas,Mains Gas,216.42,C3,old_15_band,0
1,19,Ireland,Leinster,Dublin,6,NaN,Dublin 11,12,Dublin 11,2021-01-04,2021,2021Q1,2005.0,Existing,Sale,88.000,2,40.33,44.95,44.95,12.78,1.85,17.67,0.0,Pitch Roof-Insul.on Ceiling,0.37,0.26,0.26,2.00,3.00,0.15,Medium,Masonry,No,90.0,4.0,0.0,1.0,0.0,1.0,0.0,0.0,Natural vent.,0.0,90.1,0.1,2,1,0,0,None,90.1,110.0,Factory Insulated,0,0,0.0,Unknown,Mid-terrace house,Mains Gas,Mains Gas,132.19,B3,old_15_band,1
2,24,Ireland,Leinster,Dublin,6,NaN,Dublin 7,8,Dublin 7,2021-01-04,2021,2021Q1,1994.0,Existing,Sale,30.620,1,34.00,0.00,0.00,4.97,1.68,51.08,0.0,NaN,0.55,0.00,0.00,2.80,1.36,0.15,Medium,Masonry,No,100.0,4.0,0.0,1.0,0.0,2.0,0.0,1.0,Natural vent.,0.0,100.0,0.0,3,1,0,0,None,100.0,135.0,Factory Insulated,0,0,0.0,Unknown,Mid-floor apartment,Electricity,Electricity,257.62,D1,old_15_band,0
3,25,Ireland,Leinster,Dublin,6,NaN,Dublin 3,3,Dublin 3,2021-01-04,2021,2021Q1,1900.0,Existing,Grant Support,55.432,2,52.74,27.35,27.72,5.99,1.99,34.26,0.0,Pitch Roof-Insul.on Ceiling,1.33,1.65,0.51,2.03,2.40,0.15,Medium-low,Masonry,Yes (Unsealed),88.0,4.0,0.0,1.0,0.0,4.0,0.0,0.0,Natural vent.,0.0,91.0,0.1,3,1,0,0,"Instantaneous, without keep-ho",91.0,0.0,None,0,0,0.0,Unknown,End of terrace house,Mains Gas,Mains Gas,279.50,D2,old_15_band,0
4,27,Ireland,Leinster,Dublin,6,NaN,Dublin 9,10,Dublin 9,2021-01-04,2021,2021Q1,1955.0,Existing,Sale,138.290,3,52.11,67.79,68.24,17.81,1.85,37.01,0.0,Pitch Roof-Insul.on Rafter,1.65,1.65,0.25,2.57,3.00,0.15,Medium,Masonry,No,100.0,4.0,0.0,1.0,0.0,3.0,0.0,0.0,Natural vent.,0.0,90.1,0.1,3,1,0,0,None,90.1,98.0,Factory Insulated,0,0,0.0,Unknown,Mid-terrace house,Mains Gas,Mains Gas,201.59,C3,old_15_band,0


In [19]:
# Saving the clean Dublin BER table.

save_folder = dublin_clean + r'\ber'
os.makedirs(save_folder, exist_ok=True)

ber.to_csv(save_folder + r'\ber_dublin_clean.csv', index=False)
print('Saved', ber.shape, 'to', save_folder + r'\ber_dublin_clean.csv')

Saved (181195, 65) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\just_dublin_dataset\Just_dublin_clean_dataset\ber\ber_dublin_clean.csv
